In [ ]:
%pip install "transformers==4.57.6" "datasets==5.0.1" \
    "accelerate==1.15.0" "pythainlp==5.3.8" \
    "sentencepiece==0.2.1" --quiet

# Exercise — Module M3: Thai sentiment fine-tune

**วิธีทำ:** เติมช่อง `# TODO` ให้ทุกเซลล์รันผ่าน แล้วตอบคำถามในช่อง **คำตอบ** — รันบน Kaggle GPU (T4) และ Internet: On เหมือน book ทุกประการ

**Dataset:** Wisesight Sentiment — <https://huggingface.co/datasets/pythainlp/wisesight_sentiment>, **License: CC0 1.0** · **Model:** mBERT (`google-bert/bert-base-multilingual-cased`), **License: Apache-2.0**

แล็บนี้ใช้ `N_TRAIN = 4_000`, `N_EPOCHS = 2` เพื่อให้เทรนจบใน ~3–5 นาที — ตัวเลขผลลัพธ์จะต่างจาก book (8k/3 epochs) เล็กน้อย ไม่เป็นไร ให้ตอบจากตัวเลขที่คุณได้จริง

## 0. เตรียมข้อมูล + เทรน (ให้มาแล้ว — รันผ่าน)

โค้ดชุดนี้คือส่วนสำคัญของ book ย่อมาไว้ด้วยกัน: โหลด Wisesight → subsample 4,000 ข้อความ → tokenize (รันเซลล์นี้ผ่านก่อน)

In [ ]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np  # noqa: E402
import torch  # noqa: E402
import transformers  # noqa: E402

transformers.set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch",
      torch.__version__)
print("device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: ไม่มี GPU — บน CPU จะช้ามาก ให้รันบน Kaggle GPU")

N_TRAIN = 4_000
N_EPOCHS = 2

import matplotlib.pyplot as plt  # noqa: E402

from datasets import load_dataset  # noqa: E402

ds = load_dataset("pythainlp/wisesight_sentiment")
LABELS = ds["train"].features["category"].names
print(ds)
print("labels:", LABELS)

from transformers import AutoTokenizer  # noqa: E402

CHECKPOINT = "google-bert/bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
print(tokenizer.tokenize("การสอบเทียบเครื่องมือวัด")[:14])

from sklearn.model_selection import train_test_split  # noqa: E402

# subsample แบบ stratified ตาม class — จบเร็วขึ้นแต่สัดส่วนคลาสเดิม
y_all = np.array(ds["train"]["category"])
idx_sub, _ = train_test_split(
    np.arange(len(y_all)),
    train_size=N_TRAIN,
    stratify=y_all,
    random_state=SEED,
)
idx_probe, _ = train_test_split(
    np.arange(len(y_all)),
    train_size=1_000,
    stratify=y_all,
    random_state=SEED,
)

splits = {
    "train": ds["train"].select(idx_sub),
    "train_probe": ds["train"].select(idx_probe),
    "val": ds["validation"],
    "test": ds["test"],
}


def tokenize(batch):
    return tokenizer(batch["texts"], truncation=True, max_length=64)


tok = {}
for name, d in splits.items():
    d = d.rename_column("category", "labels")
    tok[name] = d.map(tokenize, batched=True)
print({name: len(d) for name, d in tok.items()})

ต่อ: โหลด mBERT, fine-tune 4_000 ข้อความ × 2 epochs (ใช้เวลา ~3–5 นาทีบน T4 — รอจนจบ) แล้วทำนายผลบน test set — จะได้ `trainer`, `tok`, `y_true`, `y_pred`, `texts_test` ไว้ใช้ทุกข้อ

In [ ]:
from transformers import (  # noqa: E402
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

id2label = {i: name for i, name in enumerate(LABELS)}
label2id = {name: i for i, name in enumerate(LABELS)}
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    output_dir="m3_wisesight",
    seed=SEED,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=N_EPOCHS,
    weight_decay=0.01,
    fp16=DEVICE == "cuda",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}


trainer = Trainer(
    model,
    args,
    train_dataset=tok["train"],
    eval_dataset={
        "train_probe": tok["train_probe"],
        "val": tok["val"],
    },
    data_collator=DataCollatorWithPadding(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

from sklearn.metrics import classification_report  # noqa: E402

pred_out = trainer.predict(tok["test"])
y_pred = np.argmax(pred_out.predictions, axis=-1)
y_true = pred_out.label_ids
texts_test = ds["test"]["texts"]
print(classification_report(
    y_true, y_pred,
    labels=range(len(LABELS)), target_names=LABELS, digits=3,
    zero_division=0,
))

loss ราย epoch ของการเทรนนี้ (ไว้อ้างอิงข้อ 2):

In [ ]:
history = trainer.state.log_history
epochs = [h["epoch"] for h in history if "loss" in h]
train_loss = [h["loss"] for h in history if "loss" in h]
probe_loss = [
    h["eval_train_probe_loss"] for h in history
    if "eval_train_probe_loss" in h
]
val_loss = [
    h["eval_val_loss"] for h in history if "eval_val_loss" in h
]
val_acc = [
    h["eval_val_accuracy"] for h in history
    if "eval_val_accuracy" in h
]
print("train loss ราย epoch :", [round(x, 4) for x in train_loss])
print("val loss   ราย epoch :", [round(x, 4) for x in val_loss])
print("val accuracy ราย epoch:", [round(x, 4) for x in val_acc])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, train_loss, "o-", label="train loss")
ax.plot(epochs, probe_loss, "s--", label="train_probe loss")
ax.plot(epochs, val_loss, "^--", label="val loss")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("loss ลดลงราย epoch ระหว่าง fine-tune")
ax.legend()
plt.show()

และเครื่องมือ embedding ที่ให้มาแล้ว — ฟังก์ชัน `embed()` + เวกเตอร์ `E` ของตัวอย่าง test 500 ตัวอย่าง (ใช้กับข้อ 4):

In [ ]:
from sklearn.decomposition import PCA  # noqa: E402

rng = np.random.default_rng(SEED)
emb_idx = rng.choice(len(y_true), size=500, replace=False)
emb_texts = [texts_test[i] for i in emb_idx]
emb_labels = [LABELS[y_true[i]] for i in emb_idx]


MODEL_DEV = next(trainer.model.parameters()).device


@torch.no_grad()
def embed(texts):
    """เวกเตอร์ [CLS] จากชั้นสุดท้ายของโมเดลที่เทรนแล้ว."""
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    ).to(MODEL_DEV)
    out = trainer.model(**enc, output_hidden_states=True)
    cls = out.hidden_states[-1][:, 0]
    return cls.float().cpu().numpy()


E = embed(emb_texts)
pca = PCA(n_components=2)
xy = pca.fit_transform(E)

fig, ax = plt.subplots(figsize=(6, 5))
for ci, cname in enumerate(LABELS):
    m = [j for j, y in enumerate(emb_labels) if y == cname]
    ax.scatter(xy[m, 0], xy[m, 1], s=10, alpha=0.5, label=cname)
ax.set_title("embedding (CLS) ของ 500 ข้อความ — PCA 2D")
ax.legend()
plt.show()

## ข้อ 1 — ตัดคำ: ประโยคของคุณเอง

เลือกประโยคไทย 3 ประโยคของคุณเอง (ประโยคเกี่ยวกับงานวัด/งานห้องแล็บก็ได้) แล้วเทียบ `split(" ")` กับ `word_tokenize` ค่าเริ่มต้น (`newmm`) และ `engine="longest"`

**คำถาม:** ประโยคไหนที่สอง engine ตัดไม่เหมือนกัน — และแบบไหนถูกกว่าตามความรู้สึกคุณ?

In [ ]:
from pythainlp.tokenize import word_tokenize  # noqa: E402, F401

my_sentences = [
    # TODO 1: ใส่ประโยคไทยของคุณ 3 ประโยค
    "...",
    "...",
    "...",
]

# TODO 1: วนลูปพิมพ์ s.split(" "), word_tokenize(s) และ
# word_tokenize(s, engine="longest") ของทุกประโยค

**คำตอบข้อ 1:**

_(เติมตรงนี้)_

## ข้อ 2 — class ไหนที่โมเดล "อ่อน" ที่สุด?

เขียนโค้ดหา **class ที่มี recall ต่ำสุด** จากผลบน test set (คำใบ้: `classification_report(..., output_dict=True)` แล้ววนลูป `LABELS`)

**คำถาม:** class นั้นมี recall กี่ % — แปลว่าอะไรเป็นภาษาคน?

In [ ]:
from sklearn.metrics import classification_report  # noqa: E402

report = classification_report(
    y_true, y_pred,
    labels=range(len(LABELS)), target_names=LABELS, digits=3,
    zero_division=0, output_dict=True,
)

# TODO 2: หาชื่อ class ที่มี recall ต่ำสุด เก็บใน weakest
# แล้วพิมพ์ recall ของ class นั้น + จำนวนตัวอย่างจริง
# (คำใบ้: LABELS.index(weakest) ได้ index; (y_true == ci).sum())

**คำตอบข้อ 2:**

_(เติมตรงนี้ — recall ต่ำสุด = class ___ ที่ ___ % แปลว่า ...)_

## ข้อ 3 — Error inspection ของ class ที่อ่อนที่สุด

พิมพ์ **5 ตัวอย่างแรก**ที่ "จริง = class ที่ recall ต่ำสุด แต่โมเดลตอบผิด" แล้วเขียนสั้น ๆ ว่าเห็น pattern อะไร (เช่น การบ่นแบบไม่มีคำหยาบ, คำถามไม่มี ?)

In [ ]:
# TODO 3: หา index ของตัวอย่างที่ y_true == index ของ weakest
# แต่ y_pred != index ของ weakest แล้วพิมพ์ 5 ตัวอย่างแรก
# (พิมพ์ class ที่โมเดลตอบด้วย: LABELS[y_pred[i]])

**คำตอบข้อ 3:**

_(เติมตรงนี้ — pattern ที่เห็น: ...)_

## ข้อ 4 — embedding + kNN กับประโยคของคุณเอง

ใช้ฟังก์ชัน `embed()` และเวกเตอร์ `E` ที่ให้มา: แปลงประโยคของคุณ 1 ประโยคเป็น embedding แล้วหาว่า "ข้อความใน test set 500 ตัวอย่าง" ตัวไหนใกล้สุด 5 ตัว

**คำถาม:** เพื่อนบ้านที่ได้ "สมเหตุสมผล" กับประโยคของคุณไหม — label ของเพื่อนบ้านตรงกับอารมณ์ประโยคคุณไหม?

In [ ]:
from sklearn.neighbors import NearestNeighbors  # noqa: E402

my_text = "TODO 4: ใส่ประโยคของคุณที่นี่"

E_my = embed([my_text])
nn2 = NearestNeighbors(n_neighbors=5).fit(E)
D_my, I_my = nn2.kneighbors(E_my)

# TODO 4: วนลูปพิมพ์ 5 เพื่อนบ้าน: dist=D_my[0][rank],
# label=emb_labels[I_my[0][rank]], ข้อความ=emb_texts[I_my[0][rank]]

**คำตอบข้อ 4:**

_(เติมตรงนี้)_

## ข้อ 5 — zero-shot กับประโยคของคุณเอง

รันเซลล์ scaffold ด้านล่าง (โหลด XLM-T — ดาวน์โหลด ~1.1 GB ครั้งแรก) แล้วใช้ `zero_shot()` กับ `my_sentences` จากข้อ 1 เทียบกับ mBERT ที่เทรนแล้ว (คำใบ้: ทำนายด้วย `tokenizer` + `trainer.model` เหมือนที่ `embed()` ทำ แต่ดึง `logits` แล้ว `argmax`)

**คำถาม:** zero-shot ตอบตรงกับความหมายจริงกี่ประโยค — supervised กี่ประโยค? (อย่าลืมว่า XLM-T ไม่มี class คำถาม — ประโยค `q` มันตอบไม่ได้)

โหลดโมเดล zero-shot (รันตอนทำข้อ 5):

In [ ]:
ZS_CHECKPOINT = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
zs_tok = AutoTokenizer.from_pretrained(ZS_CHECKPOINT)
zs_model = AutoModelForSequenceClassification.from_pretrained(
    ZS_CHECKPOINT,
).to(DEVICE)

# XLM-T ทำนาย 3 class: 0=negative 1=neutral 2=positive
# Wisesight เรียง 0=pos 1=neu 2=neg 3=q — ตารางแปลงดังนี้
ZS_MAP = {0: 2, 1: 1, 2: 0}


@torch.no_grad()
def zero_shot(texts):
    """ทำนายด้วย XLM-T แล้วแปลง id ให้ตรงลำดับ LABELS ของ Wisesight."""
    logits = []
    for start in range(0, len(texts), 32):
        enc = zs_tok(
            texts[start:start + 32],
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt",
        ).to(DEVICE)
        out = zs_model(**enc)
        logits.append(out.logits.float().cpu())
    return torch.cat(logits).argmax(-1).numpy()

In [ ]:
# TODO 5: เรียก zero_shot(my_sentences) ได้ id แบบ Wisesight
# แล้วทำนายด้วย mBERT ด้วยวิธีเดียวกัน (แต่ logits ไม่ต้องแปลง)
# พิมพ์เทียบทีละประโยค: ประโยค | mBERT ตอบ | zero-shot ตอบ

**คำตอบข้อ 5:**

_(เติมตรงนี้)_

## สรุป (เติมหลังทำครบ)

- class ที่โมเดลอ่อนที่สุด: ______ — เพราะ ______
- สิ่งที่ error inspection เผยที่ตัวเลขไม่บอก: ______
- zero-shot พอใช้ได้เมื่อ ______ แต่ supervised จำเป็นเมื่อ ______